# Revisión de datos crudos de NODEX

12 casos solicitados, una corrida por caso. Este notebook solo carga, verifica y muestra datos; la evaluación posterior queda a cargo del investigador. No requiere pandas.


In [ ]:
from pathlib import Path
import json, hashlib
from IPython.display import display
BASE = Path.cwd()
if not (BASE / 'extraction.json').exists():
    BASE = BASE / 'extracted_data' / 'enriquecido-12casos-r2-20261104'
assert (BASE / 'extraction.json').is_file(), 'Abra el notebook desde la carpeta de extracción o ajuste BASE.'
def load(name):
    path = BASE / name
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()] if path.exists() else []
def table(rows):
    try:
        import pandas as pd
    except ImportError:
        display(rows)
    else:
        display(pd.DataFrame(rows))
extraction = json.loads((BASE / 'extraction.json').read_text(encoding='utf-8'))
display(extraction)


## Integridad y estado de la ejecución
Los hashes verifican las copias; no califican predicciones. Ausente, null, lista vacía y cadena vacía permanecen diferenciados.


In [ ]:
manifest = json.loads((BASE / 'manifest.json').read_text(encoding='utf-8'))
for item in manifest:
    if item['available']:
        assert hashlib.sha256((BASE / item['export_path']).read_bytes()).hexdigest() == item['sha256_copy'] == item['sha256_source']
table(load('run_status.jsonl'))
missing = [item for item in manifest if not item['available']]
table(missing)


## E1 · Detección
Cada registro incluye identidad y puntero al original. Referencias y predicciones se muestran por separado.


In [ ]:
for name in ['detection/predictions.jsonl', 'detection/verdicts.jsonl', 'detection/ground_truth.jsonl']:
    print(name)
    table(load(name))


## E2 · Reconstrucción
Cada registro incluye identidad y puntero al original. Referencias y predicciones se muestran por separado.


In [ ]:
for name in ['reconstruction/predictions_items.jsonl', 'reconstruction/predictions.jsonl', 'reconstruction/reference.jsonl', 'reconstruction/evidence.jsonl']:
    print(name)
    table(load(name))


## E3 · Técnicas MITRE ATT&CK
Cada registro incluye identidad y puntero al original. Referencias y predicciones se muestran por separado.


In [ ]:
for name in ['mitre/attributions_items.jsonl', 'mitre/attributions.jsonl', 'mitre/ground_truth.jsonl', 'mitre/retrieved_candidates.jsonl']:
    print(name)
    table(load(name))


## E4 · Reportes
Cada registro incluye identidad y puntero al original. Referencias y predicciones se muestran por separado.


In [ ]:
for name in ['reports/text.jsonl', 'reports/sections.jsonl']:
    print(name)
    table(load(name))


## E5 · Rendimiento
Cada registro incluye identidad y puntero al original. Referencias y predicciones se muestran por separado.


In [ ]:
for name in ['performance/processes.jsonl', 'performance/runs.jsonl', 'performance/llm_calls_items.jsonl', 'performance/llm_json.jsonl', 'performance/consultas_externas_items.jsonl']:
    print(name)
    table(load(name))


## Inspección de una ejecución
Cambie `case_id` para consultar otro caso. event_id identifica eventos reconstruidos; evidence_uids identifica evidencia citada. Los candidatos RAG no son atribuciones.


In [ ]:
case_id = 'uwf_eva_visible_0004'
for name in ['reconstruction/predictions.jsonl', 'mitre/attributions.jsonl', 'reports/text.jsonl', 'performance/runs.jsonl']:
    print(name)
    display([row for row in load(name) if row['case_id'] == case_id])
